# 🎭 Emotion Recognition using Ensemble Learning

This notebook replicates the methodology from the paper:
**"Gradient Boosting, AdaBoost, and LightGBM for Emotion Recognition: A Comparative Analysis on Wearable Sensor Data"** (ICoICT 2024)

## Features
- Heart rate and movement data (accelerometer + gyroscope)
- 107 statistical features extracted from sliding windows
- Binary (happy vs sad) and Multi-class (happy, neutral, sad) classification

**Authors**: Zikri Kholifah Nur, Rifki Wijaya, Gia Septiana Wulandari
**Dataset**: https://github.com/winwin52/emotion-recognition-smartwatch

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/winwin52/emotion-recognition-smartwatch.git
%cd emotion-recognition-smartwatch
!pip install lightgbm pyyaml -q

In [ ]:
import os
import glob
import numpy as np
import yaml
from collections import defaultdict

from sklearn import metrics
from sklearn import preprocessing
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold

import lightgbm as lgb

SEED = 42
np.random.seed(SEED)

print('✅ All libraries imported successfully!')

In [ ]:
# Get feature files from each condition
features_dir = 'features'

mo_files = sorted(glob.glob(os.path.join(features_dir, 'features_mo_*')))
mu_files = sorted(glob.glob(os.path.join(features_dir, 'features_mu_*')))
mw_files = sorted(glob.glob(os.path.join(features_dir, 'features_mw_*')))

print(f'Movie condition (mo): {len(mo_files)} files')
print(f'Music condition (mu): {len(mu_files)} files')
print(f'Music+Walking condition (mw): {len(mw_files)} files')

In [ ]:
def process_condition(fnames, condition, n_estimators=100, include_neutral=False):
    """
    Process a single experimental condition and return classification results.
    """
    print('=' * 70)
    print('Condition: %s' % condition)
    print('=' * 70)

    results = {
        'condition': condition,
        'labels': [],
        'baseline': defaultdict(list),
        'logit': defaultdict(list),
        'rf': defaultdict(list),
        'adaboost': defaultdict(list),
        'gb': defaultdict(list),
        'lightgbm': defaultdict(list)
    }

    folds = 10
    repeats = 10
    rskf = RepeatedStratifiedKFold(n_splits=folds, n_repeats=repeats, random_state=SEED)

    for fname in fnames:
        print('Classifying: %s' % fname)
        label = fname.split('/')[-1]

        # Load data
        data = np.loadtxt(fname, delimiter=',')

        if not include_neutral:
            # Binary classification: remove neutral class (emotion = 0)
            data = np.delete(data, np.where(data[:, -1] == 0), axis=0)

        print('  Shape: %s, Labels: %s' % (str(data.shape), str(np.unique(data[:, -1]))))

        np.random.shuffle(data)

        x_data = data[:, :-1]
        y_data = data[:, -1]

        # Scale features
        x_data = preprocessing.scale(x_data)

        models = [
            ('baseline', 'Baseline', DummyClassifier(strategy='most_frequent')),
            ('logit', 'Logistic Regression', LogisticRegression(max_iter=1000, random_state=SEED)),
            ('rf', 'Random Forest', RandomForestClassifier(n_estimators=n_estimators, random_state=SEED)),
            ('adaboost', 'AdaBoost', AdaBoostClassifier(n_estimators=n_estimators, random_state=SEED)),
            ('gb', 'Gradient Boosting', GradientBoostingClassifier(n_estimators=n_estimators, random_state=SEED)),
            ('lightgbm', 'LightGBM', lgb.LGBMClassifier(n_estimators=n_estimators, random_state=SEED, verbose=-1)),
        ]

        results['labels'].append(label)

        for key, name, clf in models:
            scores = {'f1': [], 'acc': [], 'auc': []}

            for train_idx, test_idx in rskf.split(x_data, y_data):
                x_train, x_test = x_data[train_idx], x_data[test_idx]
                y_train, y_test = y_data[train_idx], y_data[test_idx]

                clf.fit(x_train, y_train)
                y_pred = clf.predict(x_test)
                y_proba = clf.predict_proba(x_test)

                _acc = metrics.accuracy_score(y_test, y_pred)

                if include_neutral:
                    _f1 = metrics.f1_score(y_test, y_pred, average='weighted')
                    if len(np.unique(y_test)) > 1:
                        _auc = metrics.roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
                    else:
                        _auc = 0.5
                else:
                    _f1 = metrics.f1_score(y_test, y_pred, average='binary' if len(np.unique(y_test)) == 2 else 'weighted')
                    if len(np.unique(y_test)) > 1:
                        _auc = metrics.roc_auc_score(y_test, y_proba[:, 1])
                    else:
                        _auc = 0.5

                scores['f1'].append(_f1)
                scores['acc'].append(_acc)
                scores['auc'].append(_auc)

            results[key]['f1'].append(np.mean(scores['f1']))
            results[key]['acc'].append(np.mean(scores['acc']))
            results[key]['auc'].append(np.mean(scores['auc']))

            print('    %-25s - Acc: %.4f, F1: %.4f, AUC: %.4f' % (name, np.mean(scores['acc']), np.mean(scores['f1']), np.mean(scores['auc'])))

    return results

In [ ]:
def print_summary(results, include_neutral=False):
    """Print summary statistics for all models."""
    classification_type = "Multi-class" if include_neutral else "Binary (Happy vs Sad)"

    print('\n' + '=' * 80)
    print('SUMMARY: %s Classification - %s' % (classification_type, results['condition']))
    print('=' * 80)

    print('\n%-25s %-10s %-10s %-10s' % ('Model', 'AUC', 'F1', 'Accuracy'))
    print('-' * 60)

    model_names = {
        'baseline': 'Baseline',
        'logit': 'Logistic Regression',
        'rf': 'Random Forest',
        'adaboost': 'AdaBoost',
        'gb': 'Gradient Boosting',
        'lightgbm': 'LightGBM'
    }

    for key in ['baseline', 'logit', 'rf', 'adaboost', 'gb', 'lightgbm']:
        if key in results and results[key]['auc']:
            auc = np.mean(results[key]['auc'])
            f1 = np.mean(results[key]['f1'])
            acc = np.mean(results[key]['acc'])
            print('%-25s %-10.4f %-10.4f %-10.4f' % (model_names[key], auc, f1, acc))

    print('-' * 60)

## 🏃 Run Binary Classification (Happy vs Sad)

In [ ]:
# Run binary classification for all conditions
all_results = {'binary': {}, 'neutral': {}}

# Movie condition
print('\n' + '🎬 ' * 40)
results_mo = process_condition(mo_files, 'mo', n_estimators=100, include_neutral=False)
print_summary(results_mo, include_neutral=False)
all_results['binary']['mo'] = results_mo

# Music condition
print('\n' + '🎵 ' * 40)
results_mu = process_condition(mu_files, 'mu', n_estimators=100, include_neutral=False)
print_summary(results_mu, include_neutral=False)
all_results['binary']['mu'] = results_mu

# Music+Walking condition
print('\n' + '🚶 ' * 40)
results_mw = process_condition(mw_files, 'mw', n_estimators=100, include_neutral=False)
print_summary(results_mw, include_neutral=False)
all_results['binary']['mw'] = results_mw

## 🏃 Run Multi-class Classification (Happy, Neutral, Sad)

In [ ]:
# Run multi-class classification for all conditions
print('\n' + '🎬 ' * 40)
results_mo_n = process_condition(mo_files, 'mo', n_estimators=100, include_neutral=True)
print_summary(results_mo_n, include_neutral=True)
all_results['neutral']['mo'] = results_mo_n

print('\n' + '🎵 ' * 40)
results_mu_n = process_condition(mu_files, 'mu', n_estimators=100, include_neutral=True)
print_summary(results_mu_n, include_neutral=True)
all_results['neutral']['mu'] = results_mu_n

print('\n' + '🚶 ' * 40)
results_mw_n = process_condition(mw_files, 'mw', n_estimators=100, include_neutral=True)
print_summary(results_mw_n, include_neutral=True)
all_results['neutral']['mw'] = results_mw_n

## 📊 Summary: Best Performing Models

In [ ]:
print('\n' + '=' * 80)
print('🏆 BEST PERFORMING MODELS (by Accuracy)')
print('=' * 80)

model_names = {
    'baseline': 'Baseline',
    'logit': 'Logistic Regression',
    'rf': 'Random Forest',
    'adaboost': 'AdaBoost',
    'gb': 'Gradient Boosting',
    'lightgbm': 'LightGBM'
}

print('\n📌 Binary Classification (Happy vs Sad):')
for cond_key, cond_name in [('mo', 'Movie'), ('mu', 'Music'), ('mw', 'Music+Walking')]:
    cond_results = all_results['binary'][cond_key]
    best_model = max(cond_results.keys(), key=lambda k: np.mean(cond_results[k]['acc']))
    best_acc = np.mean(cond_results[best_model]['acc'])
    print('  %s: %s (%.2f%%)' % (cond_name, model_names[best_model], best_acc * 100))

print('\n📌 Multi-class Classification (Happy, Neutral, Sad):')
for cond_key, cond_name in [('mo', 'Movie'), ('mu', 'Music'), ('mw', 'Music+Walking')]:
    cond_results = all_results['neutral'][cond_key]
    best_model = max(cond_results.keys(), key=lambda k: np.mean(cond_results[k]['acc']))
    best_acc = np.mean(cond_results[best_model]['acc'])
    print('  %s: %s (%.2f%%)' % (cond_name, model_names[best_model], best_acc * 100))

---
## 📝 Notes

Based on the original paper results:
- **LightGBM** achieved highest accuracy: **91.8%** (binary) and **84.2%** (multi-class) in Music+Walking condition
- **Gradient Boosting** also performed well: **90.4%** (binary) in Music+Walking condition

The dataset contains:
- 50 participants (predominantly female, n=43, mean age 23.18 years)
- Samsung Gear 2 smartwatches for accelerometer + gyroscope data
- Polar H7 heart rate monitors
- 3 experimental conditions with different emotional stimuli

**Feature extraction**:
- 1-second sliding windows with 50% overlap
- 24 Hz sampling rate → 24 samples per window
- 107 features per window including:
  - Accelerometer: 17 statistical features × 3 axes = 51
  - Gyroscope: 17 statistical features × 3 axes = 51
  - Angles (x, y, z axes) = 3
  - Magnitude std = 1
  - Heart rate = 1